# Prueba en vivo — BBVA MLE Engineer


 Puedes usar **cualquier herramienta**
(este notebook, tu IDE, IA, docs). Lo que evaluamos es **cómo decides y por qué**,
no que termines todo.

**Contexto:** el equipo de negocio quiere consultar las ventas en lenguaje natural.
Tú vas a: (A) revisar si los datos sirven y depurarlos lo mínimo, y
(B) montar un mini-agente LLM que responda preguntas de negocio sin alucinar.


---
## 0. Setup (ya cableado — solo ejecuta)

In [1]:
# Colab: instala dependencias
!pip -q install duckdb google-genai pandas gdown


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\angie.araque\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
# Descarga los datos de la prueba (una sola vez). Compatible con Windows y Colab.
import os
import shutil
import zipfile

FILE_ID = '1B1dC8EA01fbDdGBlhx9Z1Ex-bMXfouQV'
if not os.path.exists('data/pedidos.csv'):
    get_ipython().system(f'gdown -q {FILE_ID} -O data.zip')
    if shutil.which('unzip'):
        get_ipython().system('unzip -oq data.zip')
    else:
        with zipfile.ZipFile('data.zip') as archivo_zip:
            archivo_zip.extractall('.')
assert os.path.exists('data/pedidos.csv'), 'No encuentro data/. Revisa FILE_ID o sube data.zip a mano.'
print('Datos listos')

Datos listos


In [3]:
from pathlib import Path
import pandas as pd, duckdb

DATA_DIR = Path('data')
pedidos = pd.read_csv(DATA_DIR / 'pedidos.csv')
detalle = pd.read_csv(DATA_DIR / 'detalle_pedidos.csv')
productos = pd.read_csv(DATA_DIR / 'productos.csv')
clientes = pd.read_csv(DATA_DIR / 'clientes.csv')

print({
    'pedidos': pedidos.shape,
    'detalle': detalle.shape,
    'productos': productos.shape,
    'clientes': clientes.shape,
})

{'pedidos': (1270, 13), 'detalle': (4177, 9), 'productos': (80, 13), 'clientes': (300, 13)}


In [4]:
# DuckDB + herramienta SQL. El agente usará pedidos_limpio para ventas.
con = duckdb.connect(':memory:')
con.register('pedidos', pedidos)
con.register('detalle', detalle)
con.register('productos', productos)
con.register('clientes', clientes)

def ejecutar_sql(query: str) -> str:
    """Ejecuta una consulta SQL de solo lectura sobre las tablas analíticas."""
    consulta = query.strip()
    consulta_lower = consulta.lower()
    tablas_pii = ('email', 'telefono', 'nombre', 'apellido')
    if not (consulta_lower.startswith('select') or consulta_lower.startswith('with')):
        return 'ERROR SQL: solo se permiten consultas SELECT o WITH.'
    if ';' in consulta[:-1] or any(columna in consulta_lower for columna in tablas_pii):
        return 'ERROR SQL: consulta bloqueada por seguridad o PII.'
    try:
        df = con.execute(consulta).fetchdf()
        if df.empty:
            return 'SIN DATOS: la consulta no devolvió filas.'
        return df.head(50).to_markdown(index=False)
    except Exception as error:
        return f'ERROR SQL: {error}'

---
## PARTE A — Analítica

El negocio pregunta: **¿qué país y qué canal venden más?**

1. Explora `pedidos` y detecta **por qué esa pregunta daría un resultado no confiable** hoy.
2. Haz la **depuración mínima** necesaria y deja los datos listos (crea `pedidos_limpio`).
3. Responde en la celda de veredicto: **¿estos datos ya sirven para el agente? ¿por qué?**

### Veredicto
Sí, para este ejercicio los datos quedan utilizables después de la depuración: `pedidos_limpio` tiene una fila por pedido, fechas e importes tipados, países y canales normalizados, y no conserva ventas cuyo importe sea desconocido. Antes de producción faltaría validar con el dueño del dato la regla de deduplicación, la causa de los nulos y la frescura del origen. La tabla de clientes contiene PII, por lo que el agente no debe consultarla para identificar personas ni devolver sus datos.

In [5]:
# Diagnóstico: la agregación original no es confiable por duplicados, categorías
# inconsistentes y 51 importes netos nulos.
print('Tipos:')
print(pedidos.dtypes[['pedido_id', 'fecha_pedido', 'total_neto']])
print('\nPaíses originales:', sorted(pedidos['pais_envio'].dropna().unique().tolist()))
print('Canales originales:', sorted(pedidos['canal'].dropna().unique().tolist()))
print('Filas:', len(pedidos))
print('Pedidos únicos:', pedidos['pedido_id'].nunique())
print('Duplicados de pedido_id:', int(pedidos['pedido_id'].duplicated().sum()))
print('total_neto nulo:', int(pedidos['total_neto'].isna().sum()))

Tipos:
pedido_id           str
fecha_pedido        str
total_neto      float64
dtype: object

Países originales: [' Colombia ', 'Argentina', 'COL', 'Chile', 'Colombia', 'Ecuador', 'México', 'Perú', 'colombia']
Canales originales: ['TIENDA_FISICA', 'Tienda_Fisica', 'marketplace', 'mobile', 'tienda fisica', 'tienda_fisica', 'web']
Filas: 1270
Pedidos únicos: 1179
Duplicados de pedido_id: 91
total_neto nulo: 51


In [6]:
# Depuración mínima: normalizamos las dimensiones usadas en la pregunta,
# convertimos fecha/importe, descartamos importes sin valor y quitamos duplicados.
pedidos_limpio = pedidos.copy()
pedidos_limpio['pais_envio'] = (
    pedidos_limpio['pais_envio'].astype('string').str.strip().str.lower()
    .replace({'col': 'colombia'})
    .str.title()
)
pedidos_limpio['canal'] = (
    pedidos_limpio['canal'].astype('string').str.strip().str.lower()
    .replace({'tienda fisica': 'tienda_fisica'})
)
pedidos_limpio['fecha_pedido'] = pd.to_datetime(
    pedidos_limpio['fecha_pedido'], errors='coerce'
)
pedidos_limpio['total_neto'] = pd.to_numeric(
    pedidos_limpio['total_neto'], errors='coerce'
)
pedidos_limpio = (
    pedidos_limpio.dropna(subset=['pedido_id', 'fecha_pedido', 'pais_envio', 'canal', 'total_neto'])
    .drop_duplicates(subset=['pedido_id'], keep='last')
    .reset_index(drop=True)
)
con.register('pedidos_limpio', pedidos_limpio)

print('Filas limpias:', len(pedidos_limpio))
print('Pedidos únicos:', pedidos_limpio['pedido_id'].nunique())
print(pedidos_limpio.groupby('pais_envio', as_index=False)['total_neto'].sum().sort_values('total_neto', ascending=False))

Filas limpias: 1136
Pedidos únicos: 1136
  pais_envio    total_neto
2   Colombia  3.440999e+08
4     México  3.315343e+08
3    Ecuador  3.248688e+08
5       Perú  3.218473e+08
0  Argentina  2.948157e+08
1      Chile  2.746295e+08


---
## PARTE B —  LLM


Monta un agente que responda preguntas de negocio en lenguaje natural usando la tool
`ejecutar_sql`. Lo que evaluamos:

1. **System prompt**: cómo le das el esquema, cómo lo acotas, cómo le dices *no inventar*.
2. **Conexión agente ↔ datos**: cómo enlazas la tool y manejas errores de SQL.
3. **Anti-alucinación y PII**: que diga *no sé* si no hay datos y que **no exponga**
   email / teléfono / nombre de clientes.
4. **Velocidad**: un agente funcional lo antes posible.

In [ ]:

from google import genai
from google.genai import types
from getpass import getpass
client = genai.Client(api_key=getpass('GEMINI_API_KEY: '))

GEMINI_API_KEY: ··········


In [ ]:
SYSTEM_PROMPT = """
Eres un analista de ventas. Responde en español, de forma breve y con números claros.

Usa ejecutar_sql para toda afirmación sobre los datos; nunca inventes resultados.
Consulta siempre pedidos_limpio para métricas de pedidos/ventas. Esquema:
- pedidos_limpio(pedido_id, cliente_id, fecha_pedido, fecha_entrega, estado, canal,
  metodo_pago, pais_envio, total_bruto, descuento_pct, total_neto)
- detalle(item_id, pedido_id, producto_id, cantidad, precio_unitario, descuento_pct, subtotal)
- productos(producto_id, nombre_producto, categoria, subcategoria, precio_venta, costo,
  stock_disponible, proveedor_id, fecha_creacion, activo)
- clientes(cliente_id, ciudad, pais, segmento, fecha_registro, fecha_consentimiento, activo)

Reglas:
1. Solo genera consultas SELECT o WITH y limita resultados a 50 filas.
2. Para ventas usa SUM(total_neto); para pedidos usa COUNT(DISTINCT pedido_id).
3. Interpreta 'último mes' como el mes calendario más reciente disponible, salvo que el usuario indique otro periodo.
4. Si la herramienta devuelve SIN DATOS o ERROR SQL, dilo claramente y no completes con suposiciones.
5. Nunca selecciones ni reveles nombre, apellido, email o teléfono de clientes. Rechaza solicitudes de PII.
6. Explica el periodo y la métrica usada cuando respondas una cifra.
"""

In [ ]:
def preguntar(pregunta: str) -> str:
    if not isinstance(pregunta, str) or not pregunta.strip():
        return 'Escribe una pregunta de negocio.'
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[ejecutar_sql],
        temperature=0,
    )
    try:
        respuesta = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=pregunta,
            config=config,
        )
        return respuesta.text or 'No hay una respuesta disponible para esa pregunta.'
    except Exception as error:
        return f'No pude consultar el agente: {error}'

In [7]:
# Pruebas locales del contrato SQL (no consumen la API).
print(ejecutar_sql("SELECT pais_envio, SUM(total_neto) AS ventas FROM pedidos_limpio GROUP BY 1 ORDER BY ventas DESC LIMIT 3"))
print(ejecutar_sql("DELETE FROM pedidos_limpio"))
print(ejecutar_sql("SELECT email FROM clientes LIMIT 1"))
print(ejecutar_sql("SELECT * FROM pedidos_limpio WHERE pais_envio = 'Pais inexistente'"))

# Cuando exista GEMINI_API_KEY, habilita estas preguntas:
# print(preguntar('¿Qué país genera más ventas totales y cuánto vendió?'))
# print(preguntar('¿Cuántos pedidos están en estado entregado?'))

| pais_envio   |      ventas |
|:-------------|------------:|
| Colombia     | 3.441e+08   |
| México       | 3.31534e+08 |
| Ecuador      | 3.24869e+08 |
ERROR SQL: solo se permiten consultas SELECT o WITH.
ERROR SQL: consulta bloqueada por seguridad o PII.
SIN DATOS: la consulta no devolvió filas.


---
> Al terminar, te haremos varias preguntas de negocio en vivo para ver cómo responde
tu agente. Prepárate para **explicar cada decisión**.